# Kimi-Linear (GDN-2) — a from-scratch code-generation LLM

This notebook walks through the **entire** training-and-evaluation cycle of this
project interactively, on the **`tiny` (CPU) preset**, by importing the real
`codegen/` package and model modules — nothing is re-implemented here, so the
notebook can never drift from the source files.

What we run, end to end:

1. **Tokenizer** — train a byte-level BPE on the MBPP corpus.
2. **Data** — turn MBPP tasks into instruction-formatted, *completion-masked* batches.
3. **Model** — build the Kimi-Linear GDN-2 hybrid (linear GDN-2 + sparse MLA + MoE FFN).
4. **Train** — the real optimization loop (warmup-cosine, grad-accum, MoE balancing).
5. **Evaluate** — held-out perplexity **and** functional `pass@k` (we *execute* the
   generated code against MBPP unit tests).

> ⚠️ The `tiny` model is a *plumbing* demonstration — it exercises every component
> the real `small` GPU run uses, but a 4-layer CPU model is **not** expected to solve
> MBPP (`pass@k ≈ 0` is the expected, healthy outcome). The mirror of this notebook
> as a single script is [`smoke_test.py`](smoke_test.py).

## 0. Setup

Make sure the repo root is importable, pin JAX to CPU, and allow the sandbox to
execute generated code (it's *our own* code, run against MBPP's `assert` tests).
If JAX/Flax aren't installed yet, run `pip install -r requirements.txt` first.

In [ ]:
import os, sys, warnings

# The codegen package and the model modules live in the repo root.
sys.path.insert(0, os.path.abspath("."))

# Keep everything on CPU for the tiny cycle, and let the sandbox run unit tests.
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ["CODEGEN_ALLOW_EXEC"] = "1"
warnings.filterwarnings("ignore")

import jax
import flax.nnx as nnx
print("jax", jax.__version__, "| devices:", jax.devices())

## 1. Configuration

A single `TrainConfig` fully describes a run. We take the `tiny` preset and shrink
the step budget so the whole cycle finishes in a couple of minutes — bump
`STEPS` / `PROBLEMS` if you want a longer demo.

In [ ]:
from codegen.config import get_preset, as_dict

STEPS = 80        # training steps for this demo (the preset default is 8 epochs)
PROBLEMS = 5      # number of MBPP test problems to score pass@k on

cfg = get_preset("tiny")
cfg.max_steps = STEPS
cfg.eval_every = max(STEPS // 2, 10)
cfg.ckpt_every = STEPS
cfg.eval_max_problems = PROBLEMS

print("preset:", cfg.name)
print("model :", f"d_model={cfg.model.d_model}, n_layers={cfg.model.n_layers}, "
      f"experts={cfg.model.moe_n_routed} (top-{cfg.model.moe_top_k}), "
      f"MLA every {cfg.model.full_attn_period} layers")
print("train :", f"seq_len={cfg.train_seq_len}, batch={cfg.batch_size}, "
      f"steps={cfg.max_steps}, lr={cfg.lr}")

## 2. Tokenizer — byte-level BPE

We train a small byte-level BPE directly on the MBPP corpus. Byte-level means
**no `<unk>`** and indentation is preserved verbatim — both essential for code.

In [ ]:
from codegen.tokenizer import CodeTokenizer, train_tokenizer, _mbpp_corpus

if not os.path.exists(cfg.tokenizer_path):
    train_tokenizer(_mbpp_corpus(cfg), cfg.vocab_size, save_path=cfg.tokenizer_path)
tok = CodeTokenizer.load(cfg.tokenizer_path)
print("vocab size:", tok.vocab_size, "| pad_id:", tok.pad_id, "| eos_id:", tok.eos_id)

# Round-trip a snippet to confirm indentation survives.
sample = "def add(a, b):\n    return a + b\n"
ids = tok.encode(sample)
print("\nencoded", len(ids), "tokens; decode == original:", tok.decode(ids) == sample)

## 3. Data — instruction prompts with a completion loss mask

Each MBPP task becomes `prompt → reference solution`. The asserts are shown in the
prompt (they reveal the function name/signature); the loss applies **only** to the
solution span, so the model learns *task → code* rather than to echo the prompt.

In [ ]:
from codegen.data import load_sft_datasets, load_eval_problems

train_ds, val_ds = load_sft_datasets(cfg, tok)
print("train examples:", len(train_ds), "| val examples:", len(val_ds))

# Show how one problem is framed for the model.
prob = load_eval_problems(cfg, split="test", max_problems=1)[0]
print("\n----- prompt fed to the model -----\n")
print(prob.prompt)

In [ ]:
# The loss mask: 1 where loss is computed (the completion), 0 for prompt + padding.
ids0 = train_ds.input_ids[0]
mask0 = train_ds.loss_mask[0]
n_supervised = int(mask0.sum())
print(f"sequence length: {len(ids0)} | supervised (completion) tokens: {n_supervised}")
print("\n----- the supervised completion span (mask == 1) -----\n")
print(tok.decode([int(t) for t, m in zip(ids0, mask0) if m]))

## 4. Model — the Kimi-Linear GDN-2 hybrid

```
input_ids ─► Embed ─► [ DecoderLayer × n_layers ] ─► RMSNorm ─► LM head ─► logits
                          x += TokenMixer(RMSNorm(x))     # GDN-2 (linear) on 3/4 layers
                                                          # MLA  (full attn) every 4th
                          x += MoE(RMSNorm(x))            # DeepSeek-style sparse FFN
```

The linear **GDN-2** layers give O(L) cost with a fixed-size recurrent state (cheap
long context); the sparse **MLA** layers restore exact global lookups (matching
brackets, variable references, call signatures); the **MoE** FFN adds capacity
without proportional compute. Let's instantiate it and count parameters.

In [ ]:
from kimi_linear_gdn2 import KimiLinear

cfg.model.vocab_size = tok.vocab_size          # match the trained tokenizer
model = KimiLinear(cfg.model, rngs=nnx.Rngs(0))

import jax
params = nnx.state(model, nnx.Param)
n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"total parameters: {n_params/1e6:.2f}M")

## 5. Train

`train(cfg)` runs the real loop: AdamW with a warmup→cosine schedule, gradient
accumulation, the masked next-token loss plus the MoE auxiliary loss, and the
aux-loss-free router-bias balancing nudged toward uniform expert load every step.
It periodically evaluates and checkpoints, returning the path to the best one.
The loss should drop noticeably over the run.

In [ ]:
from codegen.train import train

best_ckpt = train(cfg)
print("\nbest checkpoint:", best_ckpt)

## 6. Reload the checkpoint and measure perplexity

We persist only the model **state**; `load_checkpoint` rebuilds the module structure
abstractly with `nnx.eval_shape` and restores the arrays into it. Perplexity on the
held-out completion tokens is cheap (no code execution).

In [ ]:
from codegen.checkpointing import load_checkpoint
from codegen.evaluate import evaluate_perplexity

model = load_checkpoint(cfg.model, best_ckpt)
ppl = evaluate_perplexity(model, val_ds, cfg.batch_size)
print(f"held-out perplexity: {ppl:.2f}")

## 7. Sample a completion

Batched temperature / top-p decoding via the model's streaming `step` (the same
path used inside `pass@k`). Here is one sampled program for the problem above.

In [ ]:
from codegen.sampling import generate_completions
from codegen.evaluate import STOP_STRINGS

completion = generate_completions(
    model, tok, prob.prompt, n_samples=1,
    stops=STOP_STRINGS,
    max_new_tokens=cfg.eval_max_new_tokens,
    temperature=cfg.eval_temperature, top_p=cfg.eval_top_p,
    key=jax.random.PRNGKey(0),
)[0]
print("----- model completion -----\n")
print(completion)

## 8. Functional `pass@k`

For each test task we sample `n_samples` programs, **execute** each against the
task's `assert` tests in a sandboxed subprocess (timeout + resource limits), count
how many pass (`c`), and report the unbiased `pass@k = 1 - C(n-c, k)/C(n, k)`
(Chen et al., 2021), averaged over tasks.

> A tiny 4-layer CPU model will almost certainly score `pass@k ≈ 0` — that is the
> expected outcome and still proves the executor + scorer work end to end.

In [ ]:
from codegen.evaluate import evaluate_pass_at_k

report = evaluate_pass_at_k(model, tok, cfg, verbose=False)
for k, v in report["pass_at_k"].items():
    print(f"pass@{k} = {v:.3f}")
print("exec status counts:", report["status_counts"])

## 9. Scaling up

Every stage above just ran end to end — the plumbing works. For a real (if modest)
run, switch to the `small` preset (`d_model=512, n_layers=12, 8 experts top-2`,
≈200M total params) on a GPU, from the command line:

```bash
python -m codegen.tokenizer --preset small               # 1. train BPE (16k)
python -m codegen.train     --preset small               # 2. train the model
CODEGEN_ALLOW_EXEC=1 \
python -m codegen.evaluate  --preset small --ckpt runs/small/ckpt-best   # 3. pass@k
```

For stronger models, add a Phase-1 pretrain on a Python corpus before MBPP SFT via
`--pretrain-corpus` (a local directory of `.py` files or a streamed HuggingFace
dataset). See the [README](README.md) for the full set of overrides.